<a href="https://colab.research.google.com/github/VakeesanM/DL-Learning-Deliverables/blob/main/Week%207%20-%20Transformers/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import torch
import torch.nn as nn
import numpy as np

In [18]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [20]:

# Basic Attention Block
class Attention(nn.Module):
    def __init__(self, embed_size, d_k):
        super(Attention, self).__init__()
        self.d_k = d_k
        self.embed_size = embed_size
        self.values = nn.Linear(self.embed_size, self.d_k, bias=False)
        self.keys = nn.Linear(self.embed_size, self.d_k, bias=False)
        self.query = nn.Linear(self.embed_size, self.d_k, bias=False)
    def forward(self, x, pad_mask):
        Q = self.query(x)
        V = self.values(x)
        K = self.keys(x)

        attn = (Q @ K.transpose(1,2)) / np.sqrt(self.d_k)
        pad_mask = pad_mask.unsqueeze(1)
        attn = attn.masked_fill(pad_mask, float('-inf'))
        attn = torch.softmax(attn, dim=-1)

        scores = attn @ V
        return scores

In [21]:
class MultiHeadAttention(nn.Module):
  def __init__(self, embed_size, heads):
    super(MultiHeadAttention, self).__init__()
    self.embed_size = embed_size
    self.d_k = self.embed_size // heads
    self.head_attn = nn.ModuleList()
    for i in range(heads):
      self.head_attn.add_module(f'Attention Head #{i}',Attention(self.embed_size, self.d_k))

    self.linear = nn.Linear(in_features=self.embed_size, out_features=self.embed_size)
  def forward(self, x, pad_mask):
    attn_outputs = []
    for head in self.head_attn:
      attn_outputs.append(head(x, pad_mask))

    x =  torch.cat(attn_outputs, dim=-1)
    x = self.linear(x)
    return x



In [22]:
class PositionalEncoding(nn.Module):
  def __init__(self, embed_size):
    super(PositionalEncoding, self).__init__()
    self.embed_size = embed_size
  def forward(self, x):
    B, S, E = x.shape # (Batch_size, Seq, Embed)

    pos = torch.arange(S).unsqueeze(1)
    i = torch.arange(E).unsqueeze(0)

    angle_rates = pos / (10000 ** (2 * (i//2) / E))

    pe = torch.zeros(S, E).to(DEVICE)
    pe[:, 0::2] = torch.sin(angle_rates[:, 0::2])
    pe[:, 1::2] = torch.cos(angle_rates[:, 1::2])

    x = x + pe.unsqueeze(0)
    return x



In [23]:
class Encoder(nn.Module):
  def __init__(self, embed_size, heads):
    super(Encoder, self).__init__()
    self.embed_size =embed_size
    self.mult_head_attn = MultiHeadAttention(embed_size, heads=heads)
    self.norm1 = nn.LayerNorm(normalized_shape=(self.embed_size))
    self.linear1 = nn.Linear(in_features=self.embed_size, out_features= self.embed_size*4)
    self.relu = nn.ReLU()
    self.linear2 = nn.Linear(in_features=self.embed_size*4, out_features= self.embed_size)
    self.norm2 = nn.LayerNorm(normalized_shape=(self.embed_size))

    # Processing Output


# Trying to keep track of tensor dim is nuking my brain.😭
  def forward(self, x, pad_mask):

    shortcut = x
    x = self.mult_head_attn(x, pad_mask)
    x = x + shortcut
    x = self.norm1(x)

    shortcut = x
    x = self.linear1(x)
    x = self.relu(x)
    x = self.linear2(x)
    x = x + shortcut
    x = self.norm2(x)

    return x


In [24]:
class SentimentClassifier(nn.Module):
  def __init__(self, vocab_size=50257, embed_size=300, heads=6, num_classes=3, blocks=6):
    super(SentimentClassifier, self).__init__()
    self.embedding = nn.Embedding(num_embeddings=50257, embedding_dim=embed_size)
    self.pos_encoding = PositionalEncoding(embed_size)
    self.encoder_blocks = nn.ModuleList()
    for i in range(blocks):
      block = Encoder(embed_size=embed_size, heads=heads)
      self.encoder_blocks.add_module(f'Encoder Block #{i}', module=block)
    self.fc = nn.Linear(in_features=embed_size, out_features=num_classes)
    # self.softmax = nn.Softmax() -  CrossEntropyLoss apparently applies Softmax

  def forward(self, x): # X's Shape is (Batch_size, Seq)
    pad_mask = (x==50256) # This is for padding
    x = self.embedding(x)  # (Batch_size, Seq, Embed)
    x.shape
    x = self.pos_encoding(x)
    for block in self.encoder_blocks:
      x = block(x, pad_mask)
    x = self.fc(x)
    x = x.mean(dim=1)
    #x = self.softmax(x)
    return x




# Loading Dataset

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import OrdinalEncoder
from transformers import AutoTokenizer

In [26]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|endoftext|>'

In [27]:
tokenizer.encode('<|endoftext|>')

[50256]

In [28]:
encoder = OrdinalEncoder()

In [29]:
data = pd.read_csv("/content/sentiment_analysis.csv")
data = data.drop(columns=["Year", "Month", "Day", "Time of Tweet", "Platform"])
data.head()

,text,sentiment
0,What a great day!!! Looks like dream.,positive
1,"I feel sorry, I miss you here in the sea beach",positive
2,Don't angry me,negative
3,We attend in the class just for listening teac...,negative
4,"Those who want to go, let them go",negative


In [30]:
X = data.iloc[:, 0]
y = encoder.fit_transform(data.iloc[:, 1].to_frame()).ravel()

In [75]:
encoder.categories_

[array(['negative', 'neutral', 'positive'], dtype=object)]

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, shuffle=True)

In [32]:
X_train = tokenizer(
    X_train.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors=None
)
X_test = tokenizer(
    X_test.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors=None
)

In [33]:
class GPT2Dataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [34]:
train_dataset = GPT2Dataset(X_train, y_train)
test_dataset  = GPT2Dataset(X_test, y_test)

In [35]:
train_loader = DataLoader(train_dataset, batch_size=25, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=25, shuffle=True)

In [36]:
len(train_loader), len(test_loader)

(18, 2)

# Training Loop

In [37]:
import time

In [38]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 29.4 MB/s eta 0:00:00


In [60]:
from torchmetrics.classification import Accuracy


In [95]:
torch.manual_seed(39)
model = SentimentClassifier().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=3e-4)
Accu_fn = Accuracy(task="multiclass", num_classes=3).to(DEVICE)

start = time.time()

EPOCHS = 15
train_loss_list = []
test_loss_list = []
Accuracies = []
for epoch in range(EPOCHS):
  model.train()
  train_loss = 0
  for batch in train_loader:
    text = batch['input_ids'].to(DEVICE)
    label = batch['labels'].to(DEVICE)
    y_preds = model(text)
    loss = loss_fn(y_preds, label)
    optim.zero_grad()
    loss.backward()
    train_loss += loss.item()
    optim.step()
  train_loss_list.append(train_loss/len(train_loader))
  model.eval()
  test_loss = 0
  accuracy = 0
  for batch in test_loader:
    text = batch['input_ids'].to(DEVICE)
    label = batch['labels'].to(DEVICE)
    with torch.inference_mode():
      y_preds = model(text)
      loss = loss_fn(y_preds, label)
      test_loss += loss.item()
      accuracy += Accu_fn(y_preds.argmax(dim=1), label)
  Accu_fn.reset()
  test_loss_list.append(test_loss/len(test_loader))
  Accuracies.append(accuracy/len(test_loader))
  print(f"Epoch: {epoch+1} | Train Loss: {train_loss_list[-1]:.3f} | Test Loss: {test_loss_list[-1]:.3f} | Accuracy: {Accuracies[-1]*100:.3f}| Time: {(time.time() - start):.2f}")

finish = time.time() - start

min = finish // 60
sec = finish % 60

print(f"Training {EPOCHS} Epochs took {min} Minutes and {sec:.0f} Seconds! ")

Epoch: 1 | Train Loss: 1.411 | Test Loss: 1.173 | Accuracy: 22.000| Time: 3.54
Epoch: 2 | Train Loss: 1.100 | Test Loss: 1.085 | Accuracy: 36.000| Time: 6.82
Epoch: 3 | Train Loss: 1.062 | Test Loss: 1.113 | Accuracy: 36.000| Time: 8.25
Epoch: 4 | Train Loss: 0.907 | Test Loss: 1.312 | Accuracy: 32.000| Time: 11.48
Epoch: 5 | Train Loss: 0.804 | Test Loss: 1.027 | Accuracy: 48.000| Time: 12.85
Epoch: 6 | Train Loss: 0.435 | Test Loss: 1.320 | Accuracy: 52.000| Time: 14.27
Epoch: 7 | Train Loss: 0.274 | Test Loss: 1.246 | Accuracy: 74.000| Time: 15.46
Epoch: 8 | Train Loss: 0.166 | Test Loss: 1.316 | Accuracy: 66.000| Time: 16.47
Epoch: 9 | Train Loss: 0.139 | Test Loss: 1.225 | Accuracy: 66.000| Time: 17.70
Epoch: 10 | Train Loss: 0.079 | Test Loss: 1.327 | Accuracy: 68.000| Time: 18.93
Epoch: 11 | Train Loss: 0.038 | Test Loss: 1.357 | Accuracy: 66.000| Time: 19.84
Epoch: 12 | Train Loss: 0.005 | Test Loss: 1.446 | Accuracy: 68.000| Time: 20.77
Epoch: 13 | Train Loss: 0.002 | Test Los

In [107]:
def Sentiment(text):
  cat = ['negative', 'neutral', 'positive']
  tokens = torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(DEVICE)
  prediction = nn.Softmax()(model(tokens)).argmax()
  print(cat[prediction.item()])

In [111]:
Sentiment("I am Happy with this!")

positive


In [112]:
Sentiment("I HATE THIS!!!!")

negative


In [119]:
Sentiment("I just love debugging so much"), Sentiment("I just LOVE debugging so much")

# Seems to recognize sarcasm on some level

positive
negative


(None, None)

In [122]:
Sentiment("Anyone know when the next update is?")

neutral
